# Belda Calm Gene Expression Microarray data simulator


## TABLE OF CONTENTS
- **1. GENERATE CALM DATA ACCORDING TO BASIC SCENARIO**
- **2. CALM DATA EVALUATION**
  - 2.1 TITLE SUBSECTION 2.1
  - 2.2. TITLE SUBSECTION 2.1


### Setup

### Updated today 06/07/26   

pip install --upgrade --no-cache-dir calm-data-generator

In [1]:
# -----------------------------------------------------------------------------------
# CONSTANTS
# Application folders
FOLDER_GRAPHS = "./.graphs/"
FOLDER_OUTPUT = "./.output/"
FOLDER_DATA = "./../.data/d_raw_insilico/"

# ------------------------
# IMPORTS
import numpy as np
import pandas as pd
from scipy.stats import pearsonr  

import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import display, Markdown, HTML

# ------------------------
# REPRODUCIBILITY
SEED = 45

# NP ARRAY READABLE OUTPUTS
float_formatter = "{:.4f}".format
np.set_printoptions(formatter={'float_kind':float_formatter},
                    suppress=True)

´
## 1. GENERATE CALM DATA ACCORDING TO BASIC SCENARIO CONFIGURATION

BASIC SCENARIO:

            ┌─────────┐          ┌─────────┐
            │   Age   │          │   Sex   │
            └───┬─────┘          └────┬────┘ 
            0.4 │    ╲            /   │ 0.4
                │     ╲       0.1/    │                   
                ▼      ╲0.3     /     ▼                   
      ┌──────────────┐  ╲      / ┌──────────────┐       ┌───────────────┐    ┌─────────────┐
      │ Genes group A│   │    │  │ Genes group B│       │ Genes group D │    │ Noise Genes │
      │    0.3-0.6   │   │    │  │    0.2-0.4   │       │    0.2-0.3    │    │             │
      └───────┬──────┘   │    │  └──────┬───────┘       └───────────────┘    └─────────────┘
          0.2 │          │    │         │ 0.5
              │          │    │         │ 
              ▼          ▼    ▼         ▼
           ┌─────────────────────────────────┐
           │       Y (diagnostic)            │
           └─────────────────────────────────┘


In [2]:
N_GENES=1000
N_SAMPLES=2000

DATA_NAME="MCY-Belda"

DEMOG_NAMES = ['age','sex','INR','batch']
TARGET_NAME = 'Y'
config_df = build_basic_config_df()
FEATURE_NAMES = generate_var_names_from_cfg( N_GENES, config_df )
GENE_NAMES = [v for v in FEATURE_NAMES if v not in DEMOG_NAMES] # ['A_gene1', ...'NS_gene1000'

# df_display(config_df)
print("FEATURE_NAMES (tail): \n", pd.Series(FEATURE_NAMES).tail(8))
print("FEATURES LENGTH = ", len(FEATURE_NAMES))

NameError: name 'build_basic_config_df' is not defined

In [ ]:
# -----------------------------------------------------------------------------------
# CALM GENERATOR
import scipy.stats as stats
from calm_data_generator.generators.clinical.Clinic import ClinicalDataGenerator
from calm_data_generator.tutorials.clinical_generator import build_correlation_matrix

gen_output = ClinicalDataGenerator(seed=SEED) 

n_demo = 4
custom_demo = {
    "INR": {"distribution": "truncnorm", "a": 0, "b": 5, "loc": 1.1, "scale": 0.3},
    "batch": {"distribution": "randint", "low": 1, "high": 5}, # 1,2,3,4
}
group_sizes = [100, 200, 300, 400]

# Y excluded from correlation.
correlations_config = [
    {"internal": (0.3, 0.6), "demo_idx": 0, "demo_corr": 0.4},   # A & A <-> age
    {"internal": (0.2, 0.4), "demo_idx": 1, "demo_corr": 0.4},   # B & B <-> sex
    {"internal": (0.2, 0.3)},                                    # D
    {"internal": 0.0},                                           # NS
]

corr_mx = build_correlation_matrix(n_demo, group_sizes, correlations_config)

datasets = gen_output.generate(
    n_samples=N_SAMPLES, # n_patients does not work
    n_genes=sum(group_sizes),
    gene_type="Microarray", 
    custom_demographic_columns=custom_demo,
    demographic_gene_correlations=corr_mx,

    target_variable_config={ "weights": {
            "Age":         0.3 * 10,   # r ≈ 0.3
            "Sex_Binario": 0.1 * 10,   # r ≈ 0.1
            "G_0":         0.2 * 10,   # r ≈ 0.2  (representative group A)
            "G_100":       0.5 * 10},  # r ≈ 0.5  (representative group B)
            "binary_threshold": "median"
        },
    )

##### **Calm Object Summary**

In [ ]:
print("##### **Calm Object Summary**")
# print("===== Calm Object Summary =====")
print(f"- datasets          : {type(datasets).__name__}")
print(f"- dataset keys      : {list(datasets.keys())}")
print(datasets['demographics'].shape) 
print(datasets['genes'].shape) 
print(f"- gene_corr_matrix  : {type(corr_mx).__name__}")
print(f"- gen_output        : {type(gen_output).__name__}")
print(corr_mx.shape)
# info = extract_clinical_generator_info(gen_output)
# print(info.keys())
# print("")

##### **Calm Object Summary**
- datasets          : dict
- dataset keys      : ['demographics', 'genes', 'proteins']
(2000, 7)
(2000, 1000)
- gene_corr_matrix  : ndarray
- gen_output        : ClinicalDataGenerator
(1004, 1004)


In [ ]:
# Extract dataframes from calm objects
demog_df = datasets['demographics'].copy()
genes_df = datasets['genes'].copy()

print("NaNs")
print(demog_df.isna().sum().sum())   # total NaNs across all columns
num = demog_df.select_dtypes(include=np.number)
print(np.isinf(num.values).sum())   # total inf values
print(genes_df.isna().sum().sum())
print(np.isinf(genes_df.values).sum())

print(f"- columns demog_df   : {demog_df.columns}")
print(f"- columns genes_df   : {genes_df.columns}")
print(demog_df.shape)
display(demog_df.tail(4))
print(genes_df.shape)
display(genes_df.head(2))

NaNs
0
0
0
0
- columns demog_df   : Index(['Age', 'Sex', 'INR', 'batch', 'Disease_Subgroup', 'Group', 'Y'], dtype='object')
- columns genes_df   : Index(['G_0', 'G_1', 'G_2', 'G_3', 'G_4', 'G_5', 'G_6', 'G_7', 'G_8', 'G_9',
       ...
       'G_990', 'G_991', 'G_992', 'G_993', 'G_994', 'G_995', 'G_996', 'G_997',
       'G_998', 'G_999'],
      dtype='object', length=1000)
(2000, 7)


,Age,Sex,INR,batch,Disease_Subgroup,Group,Y
Patient_ID,,,,,,,
PAT_67101_1996,58,Female,1.204590,2.0,Disease,Disease,-8.311112
PAT_91800_1997,64,Female,1.208303,2.0,Disease,Disease,-5.206969
PAT_62352_1998,56,Male,1.270149,1.0,Control,Control,-3.438866
PAT_21984_1999,63,Male,1.240957,2.0,Control,Control,3.897759


(2000, 1000)


,G_0,G_1,G_2,G_3,G_4,G_5,G_6,G_7,G_8,G_9,...,G_990,G_991,G_992,G_993,G_994,G_995,G_996,G_997,G_998,G_999
PAT_50150_0,6.164479,8.770295,7.182126,7.267057,6.580701,6.993091,4.819270,6.269549,7.385044,5.766068,...,6.497719,3.481613,6.990194,5.236687,5.154817,7.479032,9.838948,7.877559,7.817826,7.860356
PAT_77204_1,6.608179,11.681771,7.962278,10.776514,6.906668,7.522808,7.553889,6.684705,7.885901,6.103456,...,6.755308,2.988926,8.960865,6.091421,5.019981,5.995592,8.106340,8.025839,6.838233,7.524131


In [ ]:
gene_means = genes_df.mean(axis=0)
gene_variances  = genes_df.var(axis=0)

summary = pd.DataFrame({
    "mean": gene_means,
    "variance": gene_variances,
    "var/mean": gene_variances / gene_means
})

min_val = genes_df.min().min()
max_val = genes_df.max().max()

print("Min:", min_val)
print("Max:", max_val)
display(summary.head(2))

Min: -2.8993526349438286
Max: 17.56458914091854


,mean,variance,var/mean
G_0,6.095591,0.299236,0.049091
G_1,9.469310,1.487486,0.157085


## Arrange BELDA MCY_df

I use my feature nomenclature to reuse my functions

In [ ]:
def arrange_belda_MCY_df(demog_df: pd.DataFrame, demog_vars: list[str],
                         genes_df: pd.DataFrame, gene_vars: list[str],
                         target_var: str) -> pd.DataFrame :

    # Select demog columns from generator and rename them
    # FIX: use the real 'Y' generated via target_variable_config, not 'Group'
    # ('Group' is just the random Control/Disease propensity-score split computed
    # BEFORE genes were generated -- it has no relationship with Age/Sex/genes,
    # which is why every correlation check against it came out ~0).
    demog_df = demog_df.loc[:,['Age','Sex','INR','batch','Y']]
    # ['age','sex','INR','batch']
    demog_df.columns = demog_vars + [target_var]
    
    # DATA TRANSFORMATION OF FACTORS TO BINARY
    demog_df['sex'] = (demog_df['sex'] == 'Male').astype(int) # Male = 1
    # FIX: 'Y' is already the numeric continuous target from generate_target_variable(),
    # not the 'Group' string column -- no need to binarize against 'Disease' anymore.
    demog_df['batch'] = (demog_df['batch']).astype(int)

    df = genes_df.join( demog_df )
    df.columns = gene_vars + demog_vars + [target_var]
    
    return df
    
MCY_df = arrange_belda_MCY_df( datasets['demographics'].copy(), DEMOG_NAMES,
                               datasets['genes'].copy(), FEATURE_NAMES[0:N_GENES],
                               TARGET_NAME) 

In [ ]:
print(np.isnan(MCY_df.values).sum())
print(np.isinf(MCY_df.values).sum())
print(MCY_df[DEMOG_NAMES+[TARGET_NAME]].head())

0
0
             age  sex       INR  batch          Y
PAT_50150_0   59    0  1.295235      4   0.608793
PAT_77204_1   64    1  1.198496      4  18.765244
PAT_17802_2   61    0  1.237368      1  -9.305238
PAT_10196_3   58    1  1.459018      1   3.568543
PAT_15774_4   63    0  1.246541      2  -7.936258


`
## MCY_df correlations

In [ ]:
group_bounds, gg_tgt, gc_tgt, y_tgt = parse_config(config_df)
print_MCY_full_cor_summary(MCY_df, group_bounds, gg_tgt, gc_tgt, y_tgt, margin_error=0.05)


Constraint                       Achieved     Target   Status
--------------------------------------------------------------
  Y ↔ A                            0.3504       0.20        X
  Y ↔ B                            0.2787       0.50        X
  Y ↔ age                          0.5715       0.30        X
  Y ↔ sex                          0.3648       0.10        X
  A ↔ age                          0.4094       0.40       OK
  B ↔ sex                          0.3052       0.40        X
  A intra-GG [0.3,0.6]             0.6008       0.45        X
  B intra-GG [0.2,0.4]             0.3021       0.30       OK
  D intra-GG [0.2,0.3]             0.2341       0.25       OK
  A intra-GG [0.3,0.6]        [0.56,0.64]    [0.3,0.6]     OK
  B intra-GG [0.2,0.4]        [0.23,0.39]    [0.2,0.4]     OK
  D intra-GG [0.2,0.3]        [0.15,0.32]    [0.2,0.3]     OK


In [ ]:
check_MCY_correlations(MCY_df)


=============== FINAL CORRELATIONS ===============

==== 1. Group mean |correlation| of genes with Y_bin ====
======== Target: A≈0.2  B≈0.5  D≈0  NS≈0 ============
  A(signal)       avg|r|=0.3504  min|r|=0.3168  max|r|=0.4858
  B(signal)       avg|r|=0.2787  min|r|=0.2183  max|r|=0.7733
  D(weak)         avg|r|=0.0163  min|r|=0.0000  max|r|=0.0597
  NS(noise)       avg|r|=0.0183  min|r|=0.0000  max|r|=0.0794

==== 2. Spot-check: individual genes & covariates vs Y_bin ====
===== Target: A≈0.2  B≈0.5  age≈0.3  sex≈0.1  INR≈0  batch≈0 =====
  Variable                Cor(Y_bin)    Expected
  ------------------------------------------------
  A_gene1                     0.4858        ~0.2
  A_gene2                     0.3502        ~0.2
  B_gene102                   0.2515        ~0.5
  B_gene103                   0.2951        ~0.5
  D_gene302                  -0.0138        ~0.0
  NS_gene602                  0.0151        ~0.0
  NS_gene1000                -0.0327        ~0.0
  ----------

´
## 'Y' is independent from Matrix. Also 'sex'

# Annex. Used functions from _ease_cor.py

In [4]:
# -----------------------------------------------------------------------------------
# BUILD CONFIGURATION TABLE
def build_basic_config_df() -> pd.DataFrame:
# This could be done through an excel, but done here for transparency
    
    columns = [ "definition", "group_name", 
                                     "object_name", "minv", "maxv", "text" ]
    data = [
    # GROUP DEFINITIONS
            [   "group",      "A",   "",             1,     100,  "Group order definition"],
            [   "group",      "B",   "",             101,   300,  "Group order definition"],
            [   "group",      "D",   "",             301,   600,  "Group order definition"],
            [   "group",      "NS",  "",             601,  1000,  "Group order definition"],
    
    # DEMOGRAPHIC VARIABLES
            [   "demog",      "age",   "",            40,     85,  "Demographic variable"],
            [   "demog",      "sex",   "",             0,      1,  "Demographic variable"], # 1 - Male; 0 - Female
            [   "demog",      "INR",   "",             0,      5,  "Demographic variable"],
            [   "demog",      "batch", "",             0,      3,  "Demographic variable"],
        
    # INTRA-GROUP CORRELATIONS
            [   "corrG-G",    "A",   "A",            0.3,   0.6,  "Intra-group correlation"],
            [   "corrG-G",    "B",   "B",            0.2,   0.4,  "Intra-group correlation"],
            [   "corrG-G",    "D",   "D",            0.2,   0.3,  "Intra-group correlation"],

    # CLINICAL <-> GROUP
            [   "corrG-C",    "A",   "age",          0.4,   0.4,  "Group-Clinical correlation"],
            [   "corrG-C",    "B",   "sex",          0.4,   0.4,  "Group-Clinical correlation"],

    # Y <-> ANY
            [   "corrY-ANY",  "Y",   "A",            0.2,   0.2,  "Correlation betw. Y and other element"],
            [   "corrY-ANY",  "Y",   "B",            0.5,   0.5,  "Correlation betw. Y and other element"],
            [   "corrY-ANY",  "Y",   "age",          0.3,   0.3,  "Correlation betw. Y and other element"],
            [   "corrY-ANY",  "Y",   "sex",          0.1,   0.1,  "Correlation betw. Y and other element"],
    ]
    # no cross-group gene correlations: A <-> B = 0; A <-> D = 0
    
    df = pd.DataFrame(data, columns=columns)
    return df

# -----------------------------------------------------------------------------------
# GENERATE NAMES VECTOR FROM CONFIG TABLE
def generate_var_names_from_cfg( n_genes: int, config_df: pd.DataFrame ) -> list[str]:
    """
    Generate gene names using group definitions from config_df.
    """

    # Read group definitions from config
    group_rows = config_df[ config_df["definition"] == "group" ]

    # Generate names
    gene_names = []

    for idx in range(1, n_genes+1):
        prefix = "UNK"

        # Search gene group within the index
        for _, row in group_rows.iterrows():
        
            group_name = row["group_name"]
            start = int(row["minv"])
            end   = int(row["maxv"])

            if start <= idx <= end:     # if 1 <= idx <= 100: prefix = "A"

                prefix = group_name
                break

        gene_names.append(f"{prefix}_gene{idx}")
         
    demog_rows = config_df[config_df["definition"] == "demog"]
    var_names = gene_names + demog_rows["group_name"].tolist()

    return var_names


# -----------------------------------------------------------------------------------
# PARSE CONFIG_DF.
def parse_config(config_df: pd.DataFrame) -> tuple[dict, dict, dict, dict]:
    
    if config_df is not None:
        group_bounds = {}
        for _, row in config_df[config_df["definition"] == "group"].iterrows():
            lo = int(row["minv"]) - 1  # 1-indexed → 0-indexed
            hi = int(row["maxv"])
            group_bounds[row["group_name"]] = np.arange(lo, hi)

        gg_targets = {}
        for _, row in config_df[config_df["definition"] == "corrG-G"].iterrows():
            gg_targets[row["group_name"]] = (row["minv"], row["maxv"])

        gc_targets = {}
        for _, row in config_df[config_df["definition"] == "corrG-C"].iterrows():
            gc_targets[row["group_name"]] = (row["object_name"], row["minv"])

        y_targets = {}
        for _, row in config_df[config_df["definition"] == "corrY-ANY"].iterrows():
            y_targets[row["object_name"]] = row["minv"]
    else:
        # Hardcoded defaults matching config_basic_scene_df
        group_bounds = {
            "A":  np.arange(0,   100),
            "B":  np.arange(100, 300),
            "D":  np.arange(300, 600),
            "NS": np.arange(600, 1000),
        }
        gg_targets = {
            "A": (0.3, 0.6),
            "B": (0.2, 0.4),
            "D": (0.2, 0.3),
        }
        gc_targets = {
            "A": ("age", 0.4),
            "B": ("sex", 0.4),
        }
        y_targets = {
            "A":   0.2,
            "B":   0.5,
            "age": 0.3,
            "sex": 0.1,
        }
    return group_bounds, gg_targets, gc_targets, y_targets

# -----------------------------------------------------------------------------------
def print_MCY_full_cor_summary(df: pd.DataFrame, group_bounds: dict, gg_targets: dict,
                          gc_targets: dict, y_targets: dict, margin_error: float = 0.05):

    Y = df["Y"].values.astype(float)

    def cols_corr_with(idxs, v):
        """Pearson r of EVERY gene in idxs against vector v -> array of r."""
        block = df.iloc[:, idxs].values.astype(float)
        b = (block - block.mean(0)) / block.std(0)          # standardise columns
        vv = (v - v.mean()) / v.std()
        return (b * vv[:, None]).mean(0)                    # per-gene Pearson r

    def intra_pairs(idxs):
        """abs Pearson r of ALL unique gene-gene pairs in the group."""
        block = df.iloc[:, idxs].values.astype(float)
        C = np.corrcoef(block, rowvar=False)               # g x g matrix
        return np.abs(C[np.triu_indices_from(C, k=1)])     # upper triangle

    print(f"\n{'Constraint':<30} {'Achieved':>10} {'Target':>10} {'Status':>8}")
    print("-" * 62)

    # ---- Y correlations (mean over ALL group genes) -------------------------
    for name, r_tgt in y_targets.items():
        if name in group_bounds:
            achieved = cols_corr_with(group_bounds[name], Y).mean()
        else:
            achieved, _ = pearsonr(df[name].values.astype(float), Y)
        status = "OK" if abs(achieved - r_tgt) < margin_error else "X"
        print(f"  Y ↔ {name:<24} {achieved:>10.4f} {r_tgt:>10.2f} {status:>8}")

    # ---- G-C correlations (mean over ALL group genes) -----------------------
    for gname, (cov, r_tgt) in gc_targets.items():
        cov_vals = df[cov].values.astype(float)
        achieved = cols_corr_with(group_bounds[gname], cov_vals).mean()
        status = "OK" if abs(achieved - r_tgt) < margin_error else "X"
        print(f"  {gname} ↔ {cov:<22}   {achieved:>10.4f} {r_tgt:>10.2f} {status:>8}")

    # ---- G-G intra correlations, average over ALL pairs ---------------------
    for gname, (r_min, r_max) in gg_targets.items():
        rs = intra_pairs(group_bounds[gname])
        achieved = rs.mean()
        r_tgt = (r_min + r_max) / 2
        status = "OK" if abs(achieved - r_tgt) < margin_error else "X"
        print(f"  {gname} intra-GG [{r_min},{r_max}]       "
              f"  {achieved:>10.4f} {r_tgt:>10.2f} {status:>8}")

    # ---- G-G intra correlations (range mode), over ALL pairs ----------------
    for gname, (r_min, r_max) in gg_targets.items():
        rs = intra_pairs(group_bounds[gname])
        achieved_min, achieved_max = rs.min(), rs.max()
        status = ("OK"
                  if achieved_min >= (r_min - margin_error)
                  and achieved_max <= (r_max + margin_error)
                  else "X")
        print(f"  {gname} intra-GG [{r_min:.1f},{r_max:.1f}]"
              f"{'':<8}[{achieved_min:.2f},{achieved_max:.2f}]"
              f"{'':<4}[{r_min:.1f},{r_max:.1f}]{status:>7}")

# -----------------------------------------------------------------------------------
def check_MCY_correlations(MCY_df: pd.DataFrame,
                           n_genes: int = 1000,
                           groups: dict = None) -> None:
    """
    Validates that the simulated MCY matrix has the expected correlation structure.

    Checks:
      1. Mean |r| of each gene group with Y_bin   (A≈0.2, B≈0.5, D≈0, NS≈0)
      2. Spot-check individual genes + all covariates vs Y_bin
      3. Average inter-gene |r| within each group  (A:0.3-0.6, B:0.2-0.4, D:0.2-0.3)
      4. Covariate vs gene-group |r|               (age vs A≈0.4, sex vs B≈0.4)

    Parameters
    ----------
    MCY_df  : DataFrame with gene columns first, then covariates, last column = 'Y'
    n_genes : number of gene columns (default 1000)
    groups  : optional dict {group_name: np.array of absolute column indices}
    """

    print("\n=============== FINAL CORRELATIONS ===============")

    # Standardize only continuous numeric columns
    # Pearson r is scale-invariant, but we standardize for consistency.
    # Binary columns (Y, sex, batch) are excluded to avoid misleading z-scores.
    skip_std = {"Y", "sex", "batch"}
    cols_to_std = [c for c in MCY_df.columns if c not in skip_std]

    MCY_df = MCY_df.copy()
    MCY_df[cols_to_std] = (
        (MCY_df[cols_to_std] - MCY_df[cols_to_std].mean(axis=0))
        / MCY_df[cols_to_std].std(axis=0, ddof=1)
    )

    # Gene group index ranges — derived from n_genes, not hardcoded
    nA  = 100
    nB  = 200
    nD  = 300
    # nNS = n_genes - nA - nB - nD       # = 400 for n_genes=1000

    A_idx  = np.arange(0,        nA)
    B_idx  = np.arange(nA,       nA + nB)
    D_idx  = np.arange(nA + nB,  nA + nB + nD)
    NS_idx = np.arange(nA + nB + nD, n_genes)

    if groups is None:
        groups = {
            "A(signal)": A_idx,
            "B(signal)": B_idx,
            "D(weak)"  : D_idx,
            "NS(noise)": NS_idx,
        }
    # Gene matrix and response
    genesM = MCY_df.iloc[:, :n_genes].to_numpy()   # shape (n_samples, n_genes)
    Y_bin  = MCY_df["Y"].to_numpy()                # binary 0/1

    # 1. Group mean |r| with Y_bin  — vectorized
    print("\n==== 1. Group mean |correlation| of genes with Y_bin ====")
    print(  "======== Target: A≈0.2  B≈0.5  D≈0  NS≈0 ============")

    for group_name, idxs in groups.items():
        # Stack genes as rows → corrcoef returns (n_genes+1 × n_genes+1) matrix
        # First row/col = Y_bin; remaining = genes
        block  = np.vstack([Y_bin, genesM[:, idxs].T])   # (1+k, n_samples)
        rmat   = np.corrcoef(block)                      # (1+k, 1+k)
        r_with_Y = np.abs(rmat[0, 1:])                   # |r| of each gene with Y

        print(f"  {group_name:<16}"
              f"avg|r|={r_with_Y.mean():.4f}  "
              f"min|r|={r_with_Y.min():.4f}  "
              f"max|r|={r_with_Y.max():.4f}")

    # 2. Spot-check individual genes + ALL covariates vs Y_bin
    spot_abs = {
        "A_gene1"    : 0,
        "A_gene2"    : 1,
        "B_gene102"  : 102,
        "B_gene103"  : 103,
        "D_gene302"  : 302,
        "NS_gene602" : 602,
        "NS_gene1000": 999,
    }
    # All covariates: expected correlations with Y
    covariate_expected = {
        "age"  : "~0.3",
        "sex"  : "~0.1",
        "INR"  : "~0.0",   # confounders not linked to Y
        "batch": "~0.0",
    }

    print("\n==== 2. Spot-check: individual genes & covariates vs Y_bin ====")
    print(  "===== Target: A≈0.2  B≈0.5  age≈0.3  sex≈0.1  INR≈0  batch≈0 =====")
    print(f"  {'Variable':<22}{'Cor(Y_bin)':>12}  {'Expected':>10}")
    print("  " + "-" * 48)

    for name, col in spot_abs.items():
        r = np.corrcoef(Y_bin, genesM[:, col])[0, 1]
        group = name.split("_")[0]
        expected = {"A": "~0.2", "B": "~0.5", "D": "~0.0", "NS": "~0.0"}.get(group, "")
        print(f"  {name:<22}{r:>12.4f}  {expected:>10}")

    print("  " + "-" * 48)

    for var, expected in covariate_expected.items():
        if var not in MCY_df.columns:
            print(f"  {var:<22}{'[column not found]':>12}")
            continue
        # For binary covariates (sex, batch), use the original integer codes
        vec = MCY_df[var].to_numpy()
        r   = np.corrcoef(Y_bin, vec)[0, 1]
        print(f"  {var:<22}{r:>12.4f}  {expected:>10}")

    print("  " + "-" * 48)

    # 3. Average inter-gene |r| within each group
    #    Uses the full upper-triangle of the within-group correlation matrix.
    print("\n==== 3. Average inter-gene |r| within groups ====")
    print(  "===== Target: A:0.3-0.6  B:0.2-0.4  D:0.2-0.3  NS:~0 =====")

    for group_name, idxs in groups.items():
        sub      = genesM[:, idxs]            # (n_samples, k)
        corr_mat = np.corrcoef(sub.T)          # (k, k)
        tri      = corr_mat[np.triu_indices(len(idxs), k=1)]
        abs_tri  = np.abs(tri)
        print(f"  {group_name:<16}"
              f"avg|r|={abs_tri.mean():.4f}  "
              f"min|r|={abs_tri.min():.4f}  "
              f"max|r|={abs_tri.max():.4f}")

    # 4. Covariate vs gene-group correlations
    #    Uses groups dict so custom group definitions are respected.
    print("\n==== 4. Covariate vs gene-group correlations ====")
    print(  "===== Target: age vs A≈0.4;  sex vs B≈0.4 =====")
    print(f"  {'Covariate-Group':<24}{'avg|r|':>10}{'min|r|':>10}{'max|r|':>10}")
    print("  " + "-" * 54)

    covariate_group_pairs = [
        ("age", "A(signal)"),
        ("sex", "B(signal)"),
    ]
    for var, group_name in covariate_group_pairs:
        if var not in MCY_df.columns or group_name not in groups:
            print(f"  {var} vs {group_name:<20}  [skipped — column or group not found]")
            continue
        idxs      = groups[group_name]
        vec       = MCY_df[var].to_numpy()
        # Vectorized: stack covariate + gene block
        block     = np.vstack([vec, genesM[:, idxs].T])
        rmat      = np.corrcoef(block)
        r_vals    = np.abs(rmat[0, 1:])
        label     = f"{var} vs {group_name}"
        print(f"  {label:<24}"
              f"{r_vals.mean():>10.4f}"
              f"{r_vals.min():>10.4f}"
              f"{r_vals.max():>10.4f}")

    print("  " + "-" * 54)
